In [2]:
import pandas as pd
import numpy as np
from datetime import datetime

# Set display options for better readability
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

---

## 1. Staging: Customers Dataset

### Expected Issues:
- Date format is DD/MM/YYYY (needs conversion to datetime)
- Country names are inconsistent ("USA", "United States", "US")
- Missing values in age and gender
- Mixed case in subscription_tier

In [3]:
# Load raw customers data
customers_raw = pd.read_csv("../raw/customers.csv")

print("Raw Customers Dataset")
print("=" * 60)
print(f"Shape: {customers_raw.shape}")
print(f"\nData Types:\n{customers_raw.dtypes}")
print(f"\nMissing Values:\n{customers_raw.isna().sum()}")
print(f"\nFirst few rows:")
customers_raw.head()

Raw Customers Dataset
Shape: (5000, 7)

Data Types:
customer_id            int64
signup_date           object
country               object
age                  float64
gender                object
subscription_tier     object
monthly_fee          float64
dtype: object

Missing Values:
customer_id            0
signup_date            0
country                0
age                  250
gender               150
subscription_tier      0
monthly_fee            0
dtype: int64

First few rows:


,customer_id,signup_date,country,age,gender,subscription_tier,monthly_fee
0,1,17/10/2023,Germany,62.0,F,basic,13.00
1,2,25/04/2022,UK,NaN,M,Premium,30.48
2,3,26/01/2022,Brazil,53.0,NaN,basic,11.59
3,4,30/01/2024,Canada,26.0,M,basic,11.97
4,5,09/10/2022,Spain,44.0,F,basic,11.23


### Transformation 1: Convert signup_date to datetime

The dates are in DD/MM/YYYY format. We need to:
- Parse them correctly using `pd.to_datetime()` with the `dayfirst=True` parameter
- Store as datetime objects for easier manipulation

In [ ]:
# Create a copy to avoid modifying the original
customers_staged = customers_raw.copy()  # hceck

# Convert signup_date to datetime
customers_staged["signup_date"] = pd.to_datetime(
    customers_staged["signup_date"], dayfirst=True  # Important! Format is DD/MM/YYYY
)

# Verify the conversion
print("Before:", customers_raw["signup_date"].dtype)
print("After:", customers_staged["signup_date"].dtype)
print(f"\nSample dates:")
print(customers_staged[["customer_id", "signup_date"]].head())

Before: object
After: datetime64[ns]

Sample dates:
   customer_id signup_date
0            1  2023-10-17
1            2  2022-04-25
2            3  2022-01-26
3            4  2024-01-30
4            5  2022-10-09


### Transformation 2: Standardize country names

We have inconsistent country names that refer to the same country:
- USA, United States, US → should all be "USA"
- UK, United Kingdom, Britain → should all be "UK"

We'll create a mapping dictionary and use `.replace()`

In [5]:
# First, let's see what unique countries we have
print("Unique countries in raw data:")
print(customers_staged["country"].value_counts())

Unique countries in raw data:
country
Australia         400
USA               378
United Kingdom    377
US                368
Spain             367
Brazil            358
Britain           358
Mexico            352
France            352
United States     349
UK                344
Italy             343
Germany           336
Canada            318
Name: count, dtype: int64


In [6]:
# Create mapping dictionary for country standardization
country_mapping = {
    "United States": "USA",
    "US": "USA",
    "United Kingdom": "UK",
    "Britain": "UK",
}

# Apply the mapping
customers_staged["country"] = customers_staged["country"].replace(country_mapping)

# Verify the change
print("Standardized countries:")
print(customers_staged["country"].value_counts())

Standardized countries:
country
USA          1095
UK           1079
Australia     400
Spain         367
Brazil        358
Mexico        352
France        352
Italy         343
Germany       336
Canada        318
Name: count, dtype: int64


### Transformation 3: Handle missing values in age

**Strategy**: Fill missing ages with the median age.
- Median is more robust to outliers than mean
- For demographic data, median imputation is a reasonable approach

In [7]:
# Check missing values before
print(f"Missing ages before: {customers_staged['age'].isna().sum()}")
print(f"Median age: {customers_staged['age'].median()}")

# Fill missing ages with median
median_age = customers_staged["age"].median()
customers_staged["age"] = customers_staged["age"].fillna(median_age)

# Verify
print(f"\nMissing ages after: {customers_staged['age'].isna().sum()}")
print(f"Age statistics:\n{customers_staged['age'].describe()}")

Missing ages before: 250
Median age: 43.5

Missing ages after: 0
Age statistics:
count    5000.000000
mean       43.944000
std        14.901664
min        18.000000
25%        32.000000
50%        43.500000
75%        57.000000
max        70.000000
Name: age, dtype: float64


### Transformation 4: Handle missing values in gender

**Strategy**: Fill missing gender with "Unknown"
- Gender is categorical
- Creating an "Unknown" category preserves the information that we don't know
- Alternative: Could drop rows, but we don't want to lose customers

In [8]:
# Check missing values before
print(f"Missing genders before: {customers_staged['gender'].isna().sum()}")
print(f"Gender distribution:\n{customers_staged['gender'].value_counts(dropna=False)}")

# Fill missing gender with 'Unknown'
customers_staged["gender"] = customers_staged["gender"].fillna("Unknown")

# Verify
print(f"\nMissing genders after: {customers_staged['gender'].isna().sum()}")
print(f"Gender distribution after:\n{customers_staged['gender'].value_counts()}")

Missing genders before: 150
Gender distribution:
gender
F        2332
M        2329
Other     189
NaN       150
Name: count, dtype: int64

Missing genders after: 0
Gender distribution after:
gender
F          2332
M          2329
Other       189
Unknown     150
Name: count, dtype: int64


### Transformation 5: Standardize subscription_tier to title case

We have mixed case: "basic", "Premium", "ENTERPRISE"
Let's standardize to title case: "Basic", "Premium", "Enterprise"

In [9]:
# Check before
print("Subscription tiers before:")
print(customers_staged["subscription_tier"].value_counts())

# Standardize to title case
customers_staged["subscription_tier"] = customers_staged[
    "subscription_tier"
].str.title()

# Verify
print("\nSubscription tiers after:")
print(customers_staged["subscription_tier"].value_counts())

Subscription tiers before:
subscription_tier
basic         3008
Premium       1476
ENTERPRISE     516
Name: count, dtype: int64

Subscription tiers after:
subscription_tier
Basic         3008
Premium       1476
Enterprise     516
Name: count, dtype: int64


### Validation: Check for data quality issues

Before saving, let's validate:
1. Age is between 18-100 (reasonable range)
2. Monthly fee is positive
3. No missing values remain

In [10]:
print("Data Validation Checks")
print("=" * 60)

# Check 1: Age range
invalid_ages = customers_staged[
    (customers_staged["age"] < 18) | (customers_staged["age"] > 100)
]
print(f"✓ Customers with invalid age (< 18 or > 100): {len(invalid_ages)}")

# Check 2: Monthly fee is positive
invalid_fees = customers_staged[customers_staged["monthly_fee"] <= 0]
print(f"✓ Customers with invalid monthly fee (<= 0): {len(invalid_fees)}")

# Check 3: Missing values
print(f"\n✓ Missing values per column:")
print(customers_staged.isna().sum())

# Check 4: Data types
print(f"\n✓ Final data types:")
print(customers_staged.dtypes)

Data Validation Checks
✓ Customers with invalid age (< 18 or > 100): 0
✓ Customers with invalid monthly fee (<= 0): 0

✓ Missing values per column:
customer_id          0
signup_date          0
country              0
age                  0
gender               0
subscription_tier    0
monthly_fee          0
dtype: int64

✓ Final data types:
customer_id                   int64
signup_date          datetime64[ns]
country                      object
age                         float64
gender                       object
subscription_tier            object
monthly_fee                 float64
dtype: object


### Save staged customers dataset

In [11]:
# Save to staging folder
customers_staged.to_csv("../staging/customers_staged.csv", index=False)
print("✅ Saved customers_staged.csv")
print(f"   Shape: {customers_staged.shape}")
customers_staged.head()

✅ Saved customers_staged.csv
   Shape: (5000, 7)


,customer_id,signup_date,country,age,gender,subscription_tier,monthly_fee
0,1,2023-10-17,Germany,62.0,F,Basic,13.00
1,2,2022-04-25,UK,43.5,M,Premium,30.48
2,3,2022-01-26,Brazil,53.0,Unknown,Basic,11.59
3,4,2024-01-30,Canada,26.0,M,Basic,11.97
4,5,2022-10-09,Spain,44.0,F,Basic,11.23


---

## 2. Staging: Usage Logs Dataset

### Expected Issues:
- Date is in YYYY-MM-DD format (easier than customers!)
- Negative duration_minutes values (data entry errors)
- Need to ensure proper data types

In [12]:
# Load raw usage logs
usage_logs_raw = pd.read_csv("../raw/usage_logs.csv")

print("Raw Usage Logs Dataset")
print("=" * 60)
print(f"Shape: {usage_logs_raw.shape}")
print(f"\nData Types:\n{usage_logs_raw.dtypes}")
print(f"\nMissing Values:\n{usage_logs_raw.isna().sum()}")
print(f"\nFirst few rows:")
usage_logs_raw.head()

Raw Usage Logs Dataset
Shape: (212936, 6)

Data Types:
customer_id             int64
log_date               object
sessions                int64
duration_minutes      float64
features_used           int64
errors_encountered      int64
dtype: object

Missing Values:
customer_id           0
log_date              0
sessions              0
duration_minutes      0
features_used         0
errors_encountered    0
dtype: int64

First few rows:


,customer_id,log_date,sessions,duration_minutes,features_used,errors_encountered
0,1,2024-12-08,9,44.92,6,0
1,1,2024-11-19,1,49.53,1,0
2,1,2024-12-12,4,7.12,1,0
3,1,2024-11-14,7,44.35,6,0
4,1,2024-11-15,2,29.12,12,1


### Transformation 1: Convert log_date to datetime

In [13]:
# Create a copy
usage_logs_staged = usage_logs_raw.copy()

# Convert log_date to datetime (format is already YYYY-MM-DD)
usage_logs_staged["log_date"] = pd.to_datetime(usage_logs_staged["log_date"])

# Verify
print("Before:", usage_logs_raw["log_date"].dtype)
print("After:", usage_logs_staged["log_date"].dtype)
print(
    f"\nDate range: {usage_logs_staged['log_date'].min()} to {usage_logs_staged['log_date'].max()}"
)

Before: object
After: datetime64[ns]

Date range: 2024-10-03 00:00:00 to 2024-12-31 00:00:00


### Transformation 2: Fix negative duration values

**Strategy**: Replace negative durations with 0
- Negative durations are clearly errors
- Could be data entry mistakes (typos with minus sign)
- Setting to 0 is conservative (could also use absolute value)

In [14]:
# Check for negative durations
negative_durations = usage_logs_staged[usage_logs_staged["duration_minutes"] < 0]
print(f"Negative duration entries: {len(negative_durations)}")
print(f"Sample of negative durations:")
print(negative_durations.head())

# Replace negative durations with 0
usage_logs_staged.loc[usage_logs_staged["duration_minutes"] < 0, "duration_minutes"] = 0

# Verify
print(
    f"\nNegative durations after fix: {(usage_logs_staged['duration_minutes'] < 0).sum()}"
)
print(f"Duration statistics:\n{usage_logs_staged['duration_minutes'].describe()}")

Negative duration entries: 2128
Sample of negative durations:
     customer_id   log_date  sessions  duration_minutes  features_used  \
30             1 2024-10-27         6             -8.54             10   
139            3 2024-10-21         7             -3.72              2   
374            7 2024-11-20         8            -33.56             13   
570           12 2024-12-12         3            -72.08             14   
691           16 2024-10-05         2            -10.22              6   

     errors_encountered  
30                    0  
139                   0  
374                   1  
570                   0  
691                   0  

Negative durations after fix: 0


Duration statistics:
count    212936.000000
mean         29.755522
std          21.381151
min           0.000000
25%          14.110000
50%          24.940000
75%          40.240000
max         217.880000
Name: duration_minutes, dtype: float64


### Transformation 3: Sort by customer_id and log_date

For time series analysis, it's helpful to have data sorted chronologically

In [15]:
# Sort by customer_id and log_date
usage_logs_staged = usage_logs_staged.sort_values(
    ["customer_id", "log_date"]
).reset_index(drop=True)

print("Data sorted by customer_id and log_date")
usage_logs_staged.head(10)

Data sorted by customer_id and log_date


,customer_id,log_date,sessions,duration_minutes,features_used,errors_encountered
0,1,2024-10-04,8,29.39,9,0
1,1,2024-10-05,7,41.82,12,2
2,1,2024-10-06,5,17.94,11,1
3,1,2024-10-08,9,34.16,8,0
4,1,2024-10-09,9,45.30,4,0
5,1,2024-10-11,7,19.66,14,0
6,1,2024-10-12,1,11.08,14,0
7,1,2024-10-14,1,23.07,4,0
8,1,2024-10-15,8,20.92,8,1
9,1,2024-10-17,9,10.11,12,0


### Validation: Check data quality

In [16]:
print("Data Validation Checks")
print("=" * 60)

# Check 1: No negative values in numeric columns
print(f"✓ Negative sessions: {(usage_logs_staged['sessions'] < 0).sum()}")
print(f"✓ Negative duration: {(usage_logs_staged['duration_minutes'] < 0).sum()}")
print(f"✓ Negative features_used: {(usage_logs_staged['features_used'] < 0).sum()}")
print(f"✓ Negative errors: {(usage_logs_staged['errors_encountered'] < 0).sum()}")

# Check 2: Missing values
print(f"\n✓ Missing values per column:")
print(usage_logs_staged.isna().sum())

# Check 3: Data types
print(f"\n✓ Data types:")
print(usage_logs_staged.dtypes)

Data Validation Checks
✓ Negative sessions: 0
✓ Negative duration: 0
✓ Negative features_used: 0
✓ Negative errors: 0

✓ Missing values per column:
customer_id           0
log_date              0
sessions              0
duration_minutes      0
features_used         0
errors_encountered    0
dtype: int64

✓ Data types:
customer_id                    int64
log_date              datetime64[ns]
sessions                       int64
duration_minutes             float64
features_used                  int64
errors_encountered             int64
dtype: object


### Save staged usage logs dataset

In [17]:
# Save to staging folder
usage_logs_staged.to_csv("../staging/usage_logs_staged.csv", index=False)
print("✅ Saved usage_logs_staged.csv")
print(f"   Shape: {usage_logs_staged.shape}")
usage_logs_staged.head()

✅ Saved usage_logs_staged.csv
   Shape: (212936, 6)


,customer_id,log_date,sessions,duration_minutes,features_used,errors_encountered
0,1,2024-10-04,8,29.39,9,0
1,1,2024-10-05,7,41.82,12,2
2,1,2024-10-06,5,17.94,11,1
3,1,2024-10-08,9,34.16,8,0
4,1,2024-10-09,9,45.30,4,0


---

## 3. Staging: Support Tickets Dataset

### Expected Issues:
- Datetime format is YYYY/MM/DD HH:MM:SS
- Unresolved tickets have NaN in resolved_date (this is OK!)
- Missing satisfaction scores
- Need to calculate resolution time

In [18]:
# Load raw support tickets
support_tickets_raw = pd.read_csv("../raw/support_tickets.csv")

print("Raw Support Tickets Dataset")
print("=" * 60)
print(f"Shape: {support_tickets_raw.shape}")
print(f"\nData Types:\n{support_tickets_raw.dtypes}")
print(f"\nMissing Values:\n{support_tickets_raw.isna().sum()}")
print(f"\nFirst few rows:")
support_tickets_raw.head()

Raw Support Tickets Dataset
Shape: (5944, 7)

Data Types:
ticket_id               int64
customer_id             int64
created_date           object
resolved_date          object
category               object
priority               object
satisfaction_score    float64
dtype: object

Missing Values:
ticket_id                0
customer_id              0
created_date             0
resolved_date         1204
category                 0
priority                 0
satisfaction_score    3093
dtype: int64

First few rows:


,ticket_id,customer_id,created_date,resolved_date,category,priority,satisfaction_score
0,1,4110,2024/03/29 00:00:00,2024/03/29 08:51:44,Technical,Medium,NaN
1,2,4110,2024/03/04 00:00:00,2024/03/04 12:08:22,Billing,Low,NaN
2,3,4110,2024/03/27 00:00:00,2024/03/27 01:09:27,Technical,Medium,5.0
3,4,2990,2024/12/22 00:00:00,2024/12/23 03:33:56,Technical,Low,NaN
4,5,3341,2024/01/03 00:00:00,2024/01/05 16:12:29,Technical,High,NaN


### Transformation 1: Parse datetime columns

In [19]:
# Create a copy
support_tickets_staged = support_tickets_raw.copy()

# Parse created_date and resolved_date
support_tickets_staged["created_date"] = pd.to_datetime(
    support_tickets_staged["created_date"]
)
support_tickets_staged["resolved_date"] = pd.to_datetime(
    support_tickets_staged["resolved_date"]
)

# Verify
print("Data types after conversion:")
print(support_tickets_staged[["created_date", "resolved_date"]].dtypes)
print(
    f"\nUnresolved tickets (NaT in resolved_date): {support_tickets_staged['resolved_date'].isna().sum()}"
)

Data types after conversion:
created_date     datetime64[ns]
resolved_date    datetime64[ns]
dtype: object

Unresolved tickets (NaT in resolved_date): 1204


### Transformation 2: Handle missing satisfaction scores

**Strategy**: Fill missing satisfaction scores with 0
- 0 indicates "no feedback provided"
- This allows us to distinguish between "no feedback" and low scores
- Alternative: Could use -1 or keep as NaN

In [20]:
# Check missing satisfaction scores
print(
    f"Missing satisfaction scores: {support_tickets_staged['satisfaction_score'].isna().sum()}"
)
print(
    f"Satisfaction score distribution:\n{support_tickets_staged['satisfaction_score'].value_counts(dropna=False).sort_index()}"
)

# Fill missing with 0
support_tickets_staged["satisfaction_score"] = support_tickets_staged[
    "satisfaction_score"
].fillna(0)

# Verify
print(
    f"\nMissing satisfaction scores after: {support_tickets_staged['satisfaction_score'].isna().sum()}"
)
print(
    f"Satisfaction score distribution after:\n{support_tickets_staged['satisfaction_score'].value_counts().sort_index()}"
)

Missing satisfaction scores: 3093
Satisfaction score distribution:
satisfaction_score
1.0     119
2.0     272
3.0     501
4.0    1039
5.0     920
NaN    3093
Name: count, dtype: int64

Missing satisfaction scores after: 0
Satisfaction score distribution after:
satisfaction_score
0.0    3093
1.0     119
2.0     272
3.0     501
4.0    1039
5.0     920
Name: count, dtype: int64


### Transformation 3: Standardize category and priority to title case

In [21]:
# Standardize category and priority
support_tickets_staged["category"] = support_tickets_staged["category"].str.title()
support_tickets_staged["priority"] = support_tickets_staged["priority"].str.title()

print("Categories:", support_tickets_staged["category"].unique())
print("Priorities:", support_tickets_staged["priority"].unique())

Categories: ['Technical' 'Billing' 'General']
Priorities: ['Medium' 'Low' 'High']


### Transformation 4: Calculate resolution time in hours

For resolved tickets, calculate how long it took to resolve them

In [22]:
# Calculate resolution time (only for resolved tickets)
support_tickets_staged["resolution_time_hours"] = (
    support_tickets_staged["resolved_date"] - support_tickets_staged["created_date"]
).dt.total_seconds() / 3600  # Convert to hours

# Show statistics
resolved_tickets = support_tickets_staged[
    support_tickets_staged["resolved_date"].notna()
]
print(f"Resolution time statistics (hours):")
print(resolved_tickets["resolution_time_hours"].describe())

print(f"\nSample of resolved tickets:")
support_tickets_staged[
    ["ticket_id", "created_date", "resolved_date", "resolution_time_hours"]
].head(10)

Resolution time statistics (hours):
count    4740.000000
mean       16.098552
std        11.482414
min         0.113056
25%         7.867361
50%        13.234028
75%        21.384514
max        87.456944
Name: resolution_time_hours, dtype: float64

Sample of resolved tickets:


,ticket_id,created_date,resolved_date,resolution_time_hours
0,1,2024-03-29,2024-03-29 08:51:44,8.862222
1,2,2024-03-04,2024-03-04 12:08:22,12.139444
2,3,2024-03-27,2024-03-27 01:09:27,1.157500
3,4,2024-12-22,2024-12-23 03:33:56,27.565556
4,5,2024-01-03,2024-01-05 16:12:29,64.208056
5,6,2024-10-01,2024-10-01 04:40:00,4.666667
6,7,2024-06-02,2024-06-02 13:02:55,13.048611
7,8,2024-09-20,NaT,NaN
8,9,2024-03-29,NaT,NaN
9,10,2024-05-06,2024-05-06 09:53:15,9.887500


### Validation: Check data quality

In [23]:
print("Data Validation Checks")
print("=" * 60)

# Check 1: Resolved date should be after created date
invalid_resolution = support_tickets_staged[
    (support_tickets_staged["resolved_date"].notna())
    & (support_tickets_staged["resolved_date"] < support_tickets_staged["created_date"])
]
print(f"✓ Tickets with resolved_date before created_date: {len(invalid_resolution)}")

# Check 2: Missing values (resolved_date is OK to be missing)
print(f"\n✓ Missing values per column:")
print(support_tickets_staged.isna().sum())

# Check 3: Data types
print(f"\n✓ Data types:")
print(support_tickets_staged.dtypes)

Data Validation Checks
✓ Tickets with resolved_date before created_date: 0

✓ Missing values per column:
ticket_id                   0
customer_id                 0
created_date                0
resolved_date            1204
category                    0
priority                    0
satisfaction_score          0
resolution_time_hours    1204
dtype: int64

✓ Data types:
ticket_id                         int64
customer_id                       int64
created_date             datetime64[ns]
resolved_date            datetime64[ns]
category                         object
priority                         object
satisfaction_score              float64
resolution_time_hours           float64
dtype: object


### Save staged support tickets dataset

In [24]:
# Save to staging folder
support_tickets_staged.to_csv("../staging/support_tickets_staged.csv", index=False)
print("✅ Saved support_tickets_staged.csv")
print(f"   Shape: {support_tickets_staged.shape}")
support_tickets_staged.head()

✅ Saved support_tickets_staged.csv
   Shape: (5944, 8)


,ticket_id,customer_id,created_date,resolved_date,category,priority,satisfaction_score,resolution_time_hours
0,1,4110,2024-03-29,2024-03-29 08:51:44,Technical,Medium,0.0,8.862222
1,2,4110,2024-03-04,2024-03-04 12:08:22,Billing,Low,0.0,12.139444
2,3,4110,2024-03-27,2024-03-27 01:09:27,Technical,Medium,5.0,1.157500
3,4,2990,2024-12-22,2024-12-23 03:33:56,Technical,Low,0.0,27.565556
4,5,3341,2024-01-03,2024-01-05 16:12:29,Technical,High,0.0,64.208056


---

## 4. Staging: Churn Dataset

### Expected Issues:
- Date format in YYYY-MM-DD (straightforward)
- Churned customers have churn_date, active customers have NaN (this is correct!)

In [25]:
# Load raw churn data
churn_raw = pd.read_csv("../raw/churn.csv")

print("Raw Churn Dataset")
print("=" * 60)
print(f"Shape: {churn_raw.shape}")
print(f"\nData Types:\n{churn_raw.dtypes}")
print(f"\nMissing Values:\n{churn_raw.isna().sum()}")
print(f"\nChurn distribution:\n{churn_raw['churned'].value_counts()}")
print(f"\nFirst few rows:")
churn_raw.head()

Raw Churn Dataset
Shape: (5000, 4)

Data Types:
customer_id          int64
churn_date          object
churned              int64
observation_date    object
dtype: object

Missing Values:
customer_id            0
churn_date          3750
churned                0
observation_date       0
dtype: int64

Churn distribution:
churned
0    3750
1    1250
Name: count, dtype: int64

First few rows:


,customer_id,churn_date,churned,observation_date
0,1,NaN,0,2024-12-31
1,2,NaN,0,2024-12-31
2,3,NaN,0,2024-12-31
3,4,NaN,0,2024-12-31
4,5,NaN,0,2024-12-31


### Transformation 1: Convert date columns to datetime

In [26]:
# Create a copy
churn_staged = churn_raw.copy()

# Convert dates to datetime
churn_staged["churn_date"] = pd.to_datetime(churn_staged["churn_date"])
churn_staged["observation_date"] = pd.to_datetime(churn_staged["observation_date"])

# Verify
print("Data types after conversion:")
print(churn_staged[["churn_date", "observation_date"]].dtypes)
print(
    f"\nActive customers (NaT in churn_date): {churn_staged['churn_date'].isna().sum()}"
)
print(f"Churned customers: {churn_staged['churned'].sum()}")

Data types after conversion:
churn_date          datetime64[ns]
observation_date    datetime64[ns]
dtype: object

Active customers (NaT in churn_date): 3750
Churned customers: 1250


### Validation: Check data quality

In [27]:
print("Data Validation Checks")
print("=" * 60)

# Check 1: churned column should be 0 or 1
invalid_churn = churn_staged[~churn_staged["churned"].isin([0, 1])]
print(f"✓ Invalid churned values (not 0 or 1): {len(invalid_churn)}")

# Check 2: If churned=1, churn_date should not be NaN
churned_no_date = churn_staged[
    (churn_staged["churned"] == 1) & (churn_staged["churn_date"].isna())
]
print(f"✓ Churned customers without churn_date: {len(churned_no_date)}")

# Check 3: If churned=0, churn_date should be NaN
active_with_date = churn_staged[
    (churn_staged["churned"] == 0) & (churn_staged["churn_date"].notna())
]
print(f"✓ Active customers with churn_date: {len(active_with_date)}")

# Check 4: Churn date should be before or on observation date
invalid_dates = churn_staged[
    (churn_staged["churn_date"].notna())
    & (churn_staged["churn_date"] > churn_staged["observation_date"])
]
print(f"✓ Churn dates after observation date: {len(invalid_dates)}")

print(f"\n✓ Data types:")
print(churn_staged.dtypes)

Data Validation Checks
✓ Invalid churned values (not 0 or 1): 0
✓ Churned customers without churn_date: 0
✓ Active customers with churn_date: 0
✓ Churn dates after observation date: 0

✓ Data types:
customer_id                  int64
churn_date          datetime64[ns]
churned                      int64
observation_date    datetime64[ns]
dtype: object


### Save staged churn dataset

In [28]:
# Save to staging folder
churn_staged.to_csv("../staging/churn_staged.csv", index=False)
print("✅ Saved churn_staged.csv")
print(f"   Shape: {churn_staged.shape}")
churn_staged.head()

✅ Saved churn_staged.csv
   Shape: (5000, 4)


,customer_id,churn_date,churned,observation_date
0,1,NaT,0,2024-12-31
1,2,NaT,0,2024-12-31
2,3,NaT,0,2024-12-31
3,4,NaT,0,2024-12-31
4,5,NaT,0,2024-12-31


---

## Summary

### What We Accomplished in the Staging Layer:

✅ **Customers Dataset**:
- Converted signup_date from DD/MM/YYYY to datetime
- Standardized country names (USA, UK)
- Filled missing ages with median
- Filled missing genders with "Unknown"
- Standardized subscription_tier to title case

✅ **Usage Logs Dataset**:
- Converted log_date to datetime
- Fixed negative duration values (replaced with 0)
- Sorted by customer_id and log_date

✅ **Support Tickets Dataset**:
- Parsed created_date and resolved_date as datetime
- Filled missing satisfaction scores with 0
- Standardized category and priority to title case
- Calculated resolution_time_hours

✅ **Churn Dataset**:
- Converted churn_date and observation_date to datetime
- Validated churned column is binary

### Key Takeaways:

1. **Staging is about cleaning, not creating** - We didn't create any new features, just fixed data quality issues
2. **Different strategies for different data** - Median for numeric, "Unknown" for categorical, 0 for scores
3. **Validation is crucial** - Always check your transformations worked correctly
4. **Document your decisions** - Why did we choose median over mean? Why 0 for satisfaction?

### Next Step: Intermediate Layer

Now that we have clean, consistent data, we can move to the intermediate layer where we'll:
- Create aggregated features from usage logs
- Calculate support ticket metrics
- Join all tables together
- Engineer predictive features for churn